In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

base_path = "/Volumes/workspace/default/raw_banking"

file_names = [
    "account_statuses",
    "account_types",
    "accounts",
    "addresses",
    "branches",
    "customer_types",
    "customers",
    "loan_statuses",
    "loans",
    "transaction_types",
    "transactions",
]

dfs = {}
for name in file_names:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{base_path}/{name}.csv")
    )
    dfs[name] = df
    print(f"{name:20s} -> rows: {df.count():7d} | cols: {len(df.columns)}")

account_statuses     -> rows:       3 | cols: 2
account_types        -> rows:       5 | cols: 2
accounts             -> rows:    1667 | cols: 6
addresses            -> rows:    1222 | cols: 4
branches             -> rows:      50 | cols: 3
customer_types       -> rows:       3 | cols: 2
customers            -> rows:    1111 | cols: 6
loan_statuses        -> rows:       3 | cols: 2
loans                -> rows:     333 | cols: 7
transaction_types    -> rows:       4 | cols: 2
transactions         -> rows:   50000 | cols: 8


In [0]:
key_tables = ["transactions", "accounts", "customers", "transaction_types", "account_statuses", "account_types"]

for tbl in key_tables:
    print(f"\n==================== {tbl.upper()} ====================")
    dfs[tbl].printSchema()
    dfs[tbl].show(5, truncate=False)


==================== TRANSACTIONS ====================
root
 |-- TransactionID: integer (nullable = true)
 |-- AccountOriginID: integer (nullable = true)
 |-- AccountDestinationID: integer (nullable = true)
 |-- TransactionTypeID: integer (nullable = true)
 |-- Amount: double (nullable = true)
 |-- TransactionDate: timestamp (nullable = true)
 |-- BranchID: integer (nullable = true)
 |-- Description: string (nullable = true)

+-------------+---------------+--------------------+-----------------+-------+-------------------+--------+-----------------+
|TransactionID|AccountOriginID|AccountDestinationID|TransactionTypeID|Amount |TransactionDate    |BranchID|Description      |
+-------------+---------------+--------------------+-----------------+-------+-------------------+--------+-----------------+
|3022681      |201164         |200868              |2                |855.17 |2023-04-20 02:00:00|41      |Transaction 22681|
|3037846      |200138         |201402              |2            

In [0]:
df_txn_raw = dfs["transactions"]

# 1. Inspect exact columns and schema of transactions
df_txn_raw.printSchema()
df_txn_raw.show(5, truncate=False)

# 2. Capture raw baseline metrics for reconciliation
numeric_cols = [
    field.name for field in df_txn_raw.schema.fields
    if str(field.dataType) in ("DoubleType()", "FloatType()", "DecimalType", "IntegerType()", "LongType()")
]
print("Numeric columns in transactions:", numeric_cols)

raw_row_count = df_txn_raw.count()
print("Raw transactions row count:", raw_row_count)

root
 |-- TransactionID: integer (nullable = true)
 |-- AccountOriginID: integer (nullable = true)
 |-- AccountDestinationID: integer (nullable = true)
 |-- TransactionTypeID: integer (nullable = true)
 |-- Amount: double (nullable = true)
 |-- TransactionDate: timestamp (nullable = true)
 |-- BranchID: integer (nullable = true)
 |-- Description: string (nullable = true)

+-------------+---------------+--------------------+-----------------+-------+-------------------+--------+-----------------+
|TransactionID|AccountOriginID|AccountDestinationID|TransactionTypeID|Amount |TransactionDate    |BranchID|Description      |
+-------------+---------------+--------------------+-----------------+-------+-------------------+--------+-----------------+
|3022681      |201164         |200868              |2                |855.17 |2023-04-20 02:00:00|41      |Transaction 22681|
|3037846      |200138         |201402              |2                |806.2  |2021-08-10 15:00:00|43      |Transaction 37

In [0]:
df_accounts = dfs["accounts"]
df_customers = dfs["customers"]
df_txn_types = dfs["transaction_types"]
df_acc_statuses = dfs["account_statuses"]
df_acc_types = dfs["account_types"]
df_cust_types = dfs["customer_types"]

# First inspect exact columns of transactions and accounts so there are zero guesses
print("transactions columns:", df_txn_raw.columns)
print("accounts columns:", df_accounts.columns)

# Enrich transactions with transaction type & account/customer context
fact_transactions_enriched = (
    df_txn_raw.alias("t")
    .join(
        df_txn_types.alias("tt"),
        F.col("t.TransactionTypeID") == F.col("tt.TransactionTypeID"),
        "left"
    )
    .join(
        dim_accounts.alias("acc"),
        F.col("t.AccountOriginID") == F.col("acc.AccountID"),
        "left"
    )
    .join(
        dim_customers.alias("cust"),
        F.col("acc.CustomerID") == F.col("cust.CustomerID"),
        "left"
    )
)

print("Enriched row count:", fact_transactions_enriched.count())
fact_transactions_enriched.show(5, truncate=False)

transactions columns: ['TransactionID', 'AccountOriginID', 'AccountDestinationID', 'TransactionTypeID', 'Amount', 'TransactionDate', 'BranchID', 'Description']
accounts columns: ['AccountID', 'CustomerID', 'AccountTypeID', 'AccountStatusID', 'Balance', 'OpeningDate']
Enriched row count: 51980
+-------------+---------------+--------------------+-----------------+-------+-------------------+--------+-----------------+-----------------+----------+---------+----------+-------------+---------------+--------+-------------------+-------------+--------+---------------+----------+----------+---------+--------+--------------------------+---------+--------------+--------------+----------------+
|TransactionID|AccountOriginID|AccountDestinationID|TransactionTypeID|Amount |TransactionDate    |BranchID|Description      |TransactionTypeID|TypeName  |AccountID|CustomerID|AccountTypeID|AccountStatusID|Balance |OpeningDate        |AccountTypeID|TypeName|AccountStatusID|StatusName|CustomerID|FirstName|La

In [0]:

from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/raw_banking"

if "dfs" not in globals():
    file_names = [
        "account_statuses",
        "account_types",
        "accounts",
        "addresses",
        "branches",
        "customer_types",
        "customers",
        "loan_statuses",
        "loans",
        "transaction_types",
        "transactions",
    ]
    dfs = {
        name: (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(f"{base_path}/{name}.csv")
        )
        for name in file_names
    }

df_txn_raw = dfs["transactions"]
df_accounts = dfs["accounts"]
df_customers = dfs["customers"]
df_txn_types = dfs["transaction_types"]

# Running this to see exactly in which table count differs from distinct().count()--------------------------------

print("Raw transactions count:", df_txn_raw.count())
print("Distinct TransactionID in raw:", df_txn_raw.select("TransactionID").distinct().count())

print("Accounts count:", df_accounts.count())
print("Distinct AccountID in accounts:", df_accounts.select("AccountID").distinct().count())

print("Customers count:", df_customers.count())
print("Distinct CustomerID in customers:", df_customers.select("CustomerID").distinct().count())

print("Txn types count:", df_txn_types.count())
print("Distinct TransactionTypeID in txn_types:", df_txn_types.select("TransactionTypeID").distinct().count())

Raw transactions count: 50000
Distinct TransactionID in raw: 49500
Accounts count: 1667
Distinct AccountID in accounts: 1651
Customers count: 1111
Distinct CustomerID in customers: 1100
Txn types count: 4
Distinct TransactionTypeID in txn_types: 4


In [0]:
# 1. Inspect duplicate AccountIDs in accounts
dup_account_ids = (
    df_accounts.groupBy("AccountID")
    .count()
    .filter(F.col("count") > 1)
)
print("Duplicate AccountID groups:", dup_account_ids.count())
df_accounts.join(dup_account_ids, "AccountID", "inner").orderBy("AccountID").show(20, truncate=False)

# 2. Inspect duplicate CustomerIDs in customers
dup_customer_ids = (
    df_customers.groupBy("CustomerID")
    .count()
    .filter(F.col("count") > 1)
)
print("Duplicate CustomerID groups:", dup_customer_ids.count())
df_customers.join(dup_customer_ids, "CustomerID", "inner").orderBy("CustomerID").show(20, truncate=False)

Duplicate AccountID groups: 16
+---------+----------+-------------+---------------+--------+-------------------+-----+
|AccountID|CustomerID|AccountTypeID|AccountStatusID|Balance |OpeningDate        |count|
+---------+----------+-------------+---------------+--------+-------------------+-----+
|200002   |10839     |5            |1              |84008.77|2019-11-15 00:00:00|2    |
|200002   |10839     |5            |1              |84008.77|2019-11-15 00:00:00|2    |
|200118   |10298     |4            |1              |61554.44|2021-09-10 00:00:00|2    |
|200118   |10298     |4            |1              |61554.44|2021-09-10 00:00:00|2    |
|200147   |10931     |3            |1              |741.98  |2018-10-16 00:00:00|2    |
|200147   |10931     |3            |1              |741.98  |2018-10-16 00:00:00|2    |
|200186   |10686     |2            |1              |75229.57|2021-09-13 00:00:00|2    |
|200186   |10686     |2            |1              |75229.57|2021-09-13 00:00:00|2    |
|

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Deduplicate dimension tables on primary key using Window + ROW_NUMBER
w_acc = Window.partitionBy("AccountID").orderBy(F.col("OpeningDate").desc_nulls_last(), F.col("Balance").desc_nulls_last())
df_accounts_clean = (
    df_accounts
    .withColumn("rn", F.row_number().over(w_acc))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

w_cust = Window.partitionBy("CustomerID").orderBy(F.col("LastName").asc_nulls_last(), F.col("FirstName").asc_nulls_last())
df_customers_clean = (
    df_customers
    .withColumn("rn", F.row_number().over(w_cust))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# 2. Build clean enriched dimensions
dim_accounts = (
    df_accounts_clean.alias("a")
    .join(dfs["account_types"].alias("at"), F.col("a.AccountTypeID") == F.col("at.AccountTypeID"), "left")
    .join(dfs["account_statuses"].alias("ast"), F.col("a.AccountStatusID") == F.col("ast.AccountStatusID"), "left")
    .select(
        F.col("a.AccountID"),
        F.col("a.CustomerID"),
        F.col("a.AccountTypeID"),
        F.col("at.TypeName").alias("AccountTypeName"),
        F.col("a.AccountStatusID"),
        F.col("ast.StatusName").alias("AccountStatusName"),
        F.col("a.Balance").alias("CurrentAccountBalance"),
        F.col("a.OpeningDate"),
    )
)

dim_customers = (
    df_customers_clean.alias("c")
    .join(dfs["customer_types"].alias("ct"), F.col("c.CustomerTypeID") == F.col("ct.CustomerTypeID"), "left")
    .select(
        F.col("c.CustomerID"),
        F.col("c.FirstName"),
        F.col("c.LastName"),
        F.col("c.DateOfBirth"),
        F.col("c.AddressID"),
        F.col("c.CustomerTypeID"),
        F.col("ct.TypeName").alias("CustomerTypeName"),
    )
)

# 3. Build clean fact_transactions without duplicate join columns
fact_transactions = (
    df_txn_raw.alias("t")
    .join(df_txn_types.alias("tt"), F.col("t.TransactionTypeID") == F.col("tt.TransactionTypeID"), "left")
    .join(dim_accounts.alias("acc"), F.col("t.AccountOriginID") == F.col("acc.AccountID"), "left")
    .select(
        F.col("t.TransactionID"),
        F.col("t.AccountOriginID"),
        F.col("t.AccountDestinationID"),
        F.col("acc.CustomerID"),
        F.col("t.BranchID"),
        F.col("t.TransactionTypeID"),
        F.col("tt.TypeName").alias("TransactionTypeName"),
        F.col("t.Amount"),
        F.col("t.TransactionDate"),
        F.col("t.Description"),
    )
)

print("Clean accounts count:", dim_accounts.count())
print("Clean customers count:", dim_customers.count())
print("fact_transactions row count:", fact_transactions.count())
fact_transactions.show(5, truncate=False)

Clean accounts count: 1651
Clean customers count: 1100
fact_transactions row count: 50000
+-------------+---------------+--------------------+----------+--------+-----------------+-------------------+-------+-------------------+-----------------+
|TransactionID|AccountOriginID|AccountDestinationID|CustomerID|BranchID|TransactionTypeID|TransactionTypeName|Amount |TransactionDate    |Description      |
+-------------+---------------+--------------------+----------+--------+-----------------+-------------------+-------+-------------------+-----------------+
|3022681      |201164         |200868              |10426     |41      |2                |Withdrawal         |855.17 |2023-04-20 02:00:00|Transaction 22681|
|3037846      |200138         |201402              |10735     |43      |2                |Withdrawal         |806.2  |2021-08-10 15:00:00|Transaction 37846|
|3045293      |201002         |201180              |10458     |5       |1                |Deposit            |1229.44|2020-08